# From ODEs to PDEs - Diffusion Problems

[Full course sequence](../../ai4sci/README.md) | **4/9 · Diffusion** | Previous: [Projectile](../projectile/Getting_Started_Projectile.ipynb) | Next: [Navier–Stokes](../navier_stokes/Weather-forecasting-navier-stokes.ipynb)

This notebook targets **nvidia-physicsnemo==2.2.2**. It preserves the course problems and sequence while using the current API for models, physics residuals, losses, and training loops. A successful short run does not certify convergence or physical accuracy. Review the fixed evaluation errors in `metrics.json`, the training history in `loss.csv`, and the prediction plots together.

The `.ipynb` file provides explanations, execution cells, and result inspection; `.py` contains the actual training program; `.yaml` contains configuration. Read the examples, open the linked `.py` file in the JupyterLab editor, then **edit → save → rerun the execution cell**. Editing a Markdown code block does not change the program.

## Steady State 1D Diffusion in a Composite Bar using PINNs

### Problem Description

Our aim is to obtain the temperature distribution inside the bar that is made up of two materials with different thermal conductivity. The geometry and the problem specification of the problem can be seen below

<center><img src="images/diffusion_bar_geometry.png" alt="Drawing" style="width: 600px;"/></center>

The composite bar extends from $x=0$ to $x=2$. The bar has material of conductivity $D_1=10$ from $x=0$ to $x=1$ and $D_2=0.1$ from $x=1$ to $x=2$. Both the ends of the bar, $x=0$ and $x=2$ are maintained at constant temperatures of $0$ and $100$ respectively. For simplicity of modelling, we will treat the composite bar as two separate bars, bar 1 and bar 2, whose ends are joined together. We will treat the temperatures in the bar 1 as $U_1$ and the temperature in bar 2 as $U_2$. 

The equations and boundary conditions governing the problem can be mathematically expressed as

One dimensional diffusion of temperature in bar 1 and 2:

$$
\begin{align}
\frac{d}{dx}\left( D_1\frac{dU_1}{dx} \right) = 0, && \text{when } 0<x<1 \\
\frac{d}{dx}\left( D_2\frac{dU_2}{dx} \right) = 0, && \text{when } 1<x<2 \\
\end{align}
$$

Flux and temperature continuity at interface $(x=1)$
$$
\begin{align}
D_1\frac{dU_1}{dx} = D_2\frac{dU_2}{dx}, && \text{when } x=1 \\
U_1 = U_2, && \text{when } x=1 \\
\end{align}
$$

### Step 1: Geometry and coefficients

The domains are $[0,1]$ and $[1,2]$, with an independent MLP for each material. Sample both intervals using `torch.rand`. Preserve $D_1=10$, $D_2=0.1$, $T_a=0$, and $T_c=100$. The problem is steady state with no internal heat source.

$$T_b=\frac{T_c+(D_1/D_2)T_a}{1+D_1/D_2}.$$

The analytical solution is $T_1=xT_b+(1-x)T_a$ on the left and $T_2=(x-1)T_c+(2-x)T_b$ on the right.

### Step 2: PDEs, interface and two networks

```python
class Diffusion(PDE):
    def __init__(self, field="u", conductivity="D1"):
        self.dim = 1
        x = Symbol("x")
        u = Function(field)(x)
        d = Symbol(conductivity) if isinstance(conductivity, str) else conductivity
        self.equations = {"diffusion": -d * u.diff(x, 2)}


class DiffusionInterface(PDE):
    def __init__(self):
        self.dim = 1
        x, d1 = Symbol("x"), Symbol("D1")
        a, b = Function("u_1")(x), Function("u_2")(x)
        self.equations = {"temperature_jump": a - b,
                          "flux_jump": d1 * a.diff(x) - D2 * b.diff(x)}


class CompositeBar(torch.nn.Module):
    def __init__(self, cfg, parameterized=False):
        super().__init__()
        self.parameterized = parameterized
        self.left = mlp(2 if parameterized else 1, 1, cfg)
        self.right = mlp(2 if parameterized else 1, 1, cfg)

    def forward(self, x, d1):
        inputs = torch.cat((x - 1, (d1 - 15) / 10), dim=1) if self.parameterized else x - 1
        return 100 * self.left(inputs), 100 * self.right(inputs)
```

For the steady 1D problem, define $-D_i T_{i,xx}=0$. This is the general heat equation $T_t-\nabla\cdot(D\nabla T)-Q=0$ with time dependence and the heat source removed. At the interface, both temperature and the **gradient multiplied by thermal conductivity** must be continuous. Comparing bare gradients does not test heat-flux conservation across different materials.

### Step 3: Boundary, interior and interface losses

```python
def loss_terms(model, physics, batch_size, device):
    d1 = 5 + 20 * torch.rand(batch_size, 1, device=device) if model.parameterized else torch.full((batch_size, 1), 10.0, device=device)
    xl = torch.rand(batch_size, 1, device=device, requires_grad=True)
    xr = (1 + torch.rand(batch_size, 1, device=device)).requires_grad_()
    ul, _ = model(xl, d1)
    _, ur = model(xr, d1)
    rl = physics[0].forward({"coordinates": xl, "u_1": ul, "D1": d1})["diffusion"]
    rr = physics[1].forward({"coordinates": xr, "u_2": ur})["diffusion"]
    xi = torch.ones_like(xl, requires_grad=True)
    ui, vi = model(xi, d1)
    interface = physics[2].forward({"coordinates": xi, "u_1": ui, "u_2": vi, "D1": d1})
    at_left, _ = model(torch.zeros_like(xl), d1)
    _, at_right = model(torch.full_like(xr, 2), d1)
    return {"physics": (rl / (100 * d1)).square().mean() + (rr / (100 * D2)).square().mean(),
            "boundary": ((at_left - TA) / 100).square().mean() + ((at_right - TC) / 100).square().mean(),
            "interface_temperature": (interface["temperature_jump"] / 100).square().mean(),
            "interface_flux": (interface["flux_jump"] / (100 * d1)).square().mean()}
```

### Step 4: Validators and monitors

`heldout_before/after` computes solution and PDE RMSE at fixed coordinates separate from training samples. `temperature_jumps` and `physical_flux_jumps` evaluate the temperature jump and $D_1T_{1,x}-D_2T_{2,x}$ at x=1. Inspect the boundary values, continuity conditions, and full temperature curves together.

### Step 5: Configuration

[Fixed-D1 configuration](source_code/conf/config.yaml) · [Parameterized configuration](source_code/conf/config_param.yaml). Both files use simple YAML. Start with a short execution using the CLI `--steps` option.

### Step 6: Training

Run [diffusion_bar.py](source_code/diffusion_bar.py). A single Adam optimizer updates both networks. The temperature scale of 100 and normalization of each loss term are explicit in the code.

In [ ]:
import os, sys, json, subprocess, uuid
from pathlib import Path
import numpy as np
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tutorial").is_dir() and (p / "challenge").is_dir())
LAB = ROOT / "tutorial/diffusion_1d"
OUTPUT_BASE = Path(os.environ.get("AI4SCI_OUTPUT_DIR", str(LAB / "outputs")))
DEVICE = os.environ.get("AI4SCI_DEVICE", "cpu")
STEPS = int(os.environ.get("AI4SCI_STEPS", "200"))  # increase after the execution check
print("PhysicsNeMo target: 2.2.2", "device:", DEVICE, "steps:", STEPS)

In [ ]:
OUTPUT = OUTPUT_BASE / ("diffusion_bar-" + uuid.uuid4().hex[:8])
command = [sys.executable, str(LAB / "source_code/diffusion_bar.py"), "--device", DEVICE,
           "--steps", str(STEPS), "--seed", "42", "--output-dir", str(OUTPUT)]
subprocess.run(command, check=True, cwd=ROOT)
print(json.loads((OUTPUT / "metrics.json").read_text()))

### Visualizing the solution

In [ ]:
data = np.load(OUTPUT / "predictions.npz", allow_pickle=False)
for i, d1 in enumerate(data["D1"]):
    plt.plot(data["x"], data["reference"][i, :, 0], "--", label=f"analytical D1={d1:g}")
    plt.plot(data["x"], data["prediction"][i, :, 0], label=f"PINN D1={d1:g}")
plt.axvline(1, color="grey"); plt.xlabel("x"); plt.ylabel("temperature"); plt.legend(); plt.show()

## Parameterized 1D Diffusion of Composite Bar

As we discussed in the introductory notebook, one important advantage of a PINN solver over traditional numerical methods is its ability to solve parameterized geometries and PDEs. This was initially proposed in the [paper](https://arxiv.org/abs/1906.02382) published by Sun et al. This allows us significant computational advantage, as one can now use PINNs to solve for multiple designs/cases in a single training. Once the training is complete, it is possible to run inference on several geometry/physical parameter combinations as a post-processing step without solving the forward problem again. 

To demonstrate the concept, we will train the same 1d diffusion problem, but now by parameterizing the conductivity of the first bar in the range $(5, 25)$. Once the training is complete, we can obtain the results for any conductivity value in that range saving us the time to train multiple models.

### Parameterized model and sampling

Both MLPs take $(x,D_1)$ as input. Sample $D_1\sim U(5,25)$ for every batch. Because $D_1$ is spatially constant, define it with SymPy `Symbol("D1")` and supply its value explicitly. This removes the ambiguity of treating a string coefficient as a spatial function in the original implementation. The physical problem and parameter range are unchanged.

[diffusion_bar_parameterized.py](source_code/diffusion_bar_parameterized.py) calls the same explicit training function with parameterized=True. The plots compare D1=5,10,25; separate evaluation coordinates use D1=7.5,17.5,22.5. Assess accuracy over the continuous parameter range through sampled evaluations.

In [ ]:
OUTPUT = OUTPUT_BASE / ("diffusion_bar_parameterized-" + uuid.uuid4().hex[:8])
command = [sys.executable, str(LAB / "source_code/diffusion_bar_parameterized.py"), "--device", DEVICE,
           "--steps", str(STEPS), "--seed", "42", "--output-dir", str(OUTPUT)]
subprocess.run(command, check=True, cwd=ROOT)
print(json.loads((OUTPUT / "metrics.json").read_text()))

In [ ]:
data = np.load(OUTPUT / "predictions.npz", allow_pickle=False)
for i, d1 in enumerate(data["D1"]):
    plt.plot(data["x"], data["reference"][i, :, 0], "--", label=f"analytical D1={d1:g}")
    plt.plot(data["x"], data["prediction"][i, :, 0], label=f"PINN D1={d1:g}")
plt.xlabel("x"); plt.ylabel("temperature"); plt.legend(); plt.show()

## Visualising with ParaView

Export the same predictions to CSV, then use **Plot Data** in ParaView to compare x and temperature. The `.npz` file contains the original results; the CSV is a visualization export.

In [ ]:
np.savetxt(OUTPUT / "bar_D1_10.csv", np.column_stack((data["x"], data["prediction"][1], data["reference"][1])),
           delimiter=",", header="x,prediction,reference", comments="")
print(OUTPUT / "bar_D1_10.csv")

### Next steps

[Full course sequence](../../ai4sci/README.md) | **4/9 · Diffusion** | Previous: [Projectile](../projectile/Getting_Started_Projectile.ipynb) | Next: [Navier–Stokes](../navier_stokes/Weather-forecasting-navier-stokes.ipynb)

--- 

Don't forget to check out additional [Open Hackathons Resources](https://www.openhackathons.org/s/technical-resources) and join our [OpenACC and Hackathons Slack Channel](https://www.openacc.org/community#slack) to share your experience and get more help from the community.

---

# Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.